# PDZ Trans Count Reconciliation to FLIP Splits

This notebook reconstructs FLIP-style `PDZ:CRIPT` sequences from the two raw PDZ trans count files from Bruce's Azure folder and checks whether those reconstructed count rows map onto the public FLIP split file.

The reconstruction works as follows:

1. `trans-1.tsv`: take the first 50 amino acids from `aa_seq`, append the last 50 amino acids of wild-type PDZ, then append `:` and the final 8 amino acids from `aa_seq` as CRIPT.
2. `trans-2.tsv`: prepend the first 50 amino acids of wild-type PDZ to the first 50 amino acids from `aa_seq`, then append `:` and the final 8 amino acids from `aa_seq` as CRIPT.
3. Concatenate both sources, group by reconstructed sequence, and sum count columns.
4. Split `PDZ:CRIPT`, truncate each side at the first stop character (`*`), rejoin, and merge onto FLIP sequences.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

WT_PDZ = "GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPADLSGELRKGDQILSVNGVDLRNASHEQAAIALKNAGQTVTIIAQYKPEEYSRFEA"
PDZ_HALF_LENGTH = 50
CRIPT_LENGTH = 8

COUNT_COLUMNS = [
    "input1_e1_s0_bNA_count",
    "input2_e2_s0_bNA_count",
    "input3_e3_s0_bNA_count",
    "output1A_e1_s1_b1_count",
    "output2A_e2_s1_b1_count",
    "output3A_e3_s1_b1_count",
]

candidate_roots = [Path.cwd(), Path.cwd().parent]
repo_root = next(root for root in candidate_roots if (root / "data").exists())
data_dir = repo_root / "data"

trans1_path = data_dir / "trans-1.tsv"
trans2_path = data_dir / "trans-2.tsv"
flip_split_path = data_dir / "single_to_double_from_flip_repo.csv"
cis_path = data_dir / "cis_from_bruce_azure.tsv"


def truncate_stop(sequence: str) -> str:
    return sequence.split("*", 1)[0]


def load_pdz_counts(path: Path) -> pd.DataFrame:
    required_columns = ["aa_seq", *COUNT_COLUMNS]
    df = pd.read_csv(path, sep="\t", usecols=required_columns)
    missing_columns = set(required_columns) - set(df.columns)
    if missing_columns:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing_columns)}")
    df[COUNT_COLUMNS] = df[COUNT_COLUMNS].apply(pd.to_numeric, errors="raise").astype("int64")
    return df


def reconstruct_trans_sequences(df: pd.DataFrame, source: str) -> pd.DataFrame:
    if source == "trans-1":
        pdz = df["aa_seq"].str[:PDZ_HALF_LENGTH] + WT_PDZ[PDZ_HALF_LENGTH:]
    elif source == "trans-2":
        pdz = WT_PDZ[:PDZ_HALF_LENGTH] + df["aa_seq"].str[:PDZ_HALF_LENGTH]
    else:
        raise ValueError(f"Unexpected source: {source}")

    cript = df["aa_seq"].str[-CRIPT_LENGTH:]
    out = df.copy()
    out["source"] = source
    out["pdz_sequence_raw"] = pdz
    out["cript_sequence_raw"] = cript
    out["raw_sequence"] = out["pdz_sequence_raw"] + ":" + out["cript_sequence_raw"]
    out["pdz_sequence"] = out["pdz_sequence_raw"].map(truncate_stop)
    out["cript_sequence"] = out["cript_sequence_raw"].map(truncate_stop)
    out["sequence"] = out["pdz_sequence"] + ":" + out["cript_sequence"]
    return out


print(f"Repository root: {repo_root}")
print(f"Data directory: {data_dir}")
print(f"WT PDZ length: {len(WT_PDZ)} aa")

Repository root: c:\Users\t-renatob\OneDrive - Microsoft\Desktop\RenatoBerlinghieri\prob-protein-fitness
Data directory: c:\Users\t-renatob\OneDrive - Microsoft\Desktop\RenatoBerlinghieri\prob-protein-fitness\data
WT PDZ length: 100 aa


## File Inventory

Confirm that the expected local files are available. The trans files are the raw count source; the FLIP CSV provides the target split labels and sequence format to merge onto.

In [2]:
file_inventory = pd.DataFrame(
    [
        {
            "file": path.name,
            "exists": path.exists(),
            "size_mb": path.stat().st_size / 1e6 if path.exists() else np.nan,
        }
        for path in [trans1_path, trans2_path, flip_split_path, cis_path]
    ]
)

display(file_inventory)

,file,exists,size_mb
0,trans-1.tsv,True,39.648781
1,trans-2.tsv,True,39.998389
2,single_to_double_from_flip_repo.csv,True,0.098385
3,cis_from_bruce_azure.tsv,True,2.834332


## Load the Datasets

Load the two raw trans count files and the FLIP split table. The helper mirrors the SciStanPy `load_pdz_dataset` behavior by selecting `aa_seq` plus the six count columns.

In [3]:
trans1_raw = load_pdz_counts(trans1_path)
trans2_raw = load_pdz_counts(trans2_path)
flip_split = pd.read_csv(flip_split_path)
# Get PDZ and CRIPT sequences
flip_split[["pdz_sequence", "cript_sequence"]] = flip_split["sequence"].str.split(":", n=1, expand=True)

load_summary = pd.DataFrame(
    [
        {"dataset": "trans-1 raw counts", "rows": len(trans1_raw), "columns": len(trans1_raw.columns)},
        {"dataset": "trans-2 raw counts", "rows": len(trans2_raw), "columns": len(trans2_raw.columns)},
        {"dataset": "FLIP split", "rows": len(flip_split), "columns": len(flip_split.columns)},
    ]
)

display(load_summary)
display(flip_split.head())

,dataset,rows,columns
0,trans-1 raw counts,158655,7
1,trans-2 raw counts,159511,7
2,FLIP split,734,6


,sequence,target,set,validation,pdz_sequence,cript_sequence
0,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,-0.834694,train,False,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,
1,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,-0.085543,train,False,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,ANYKQTSV
2,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,-0.057949,train,False,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,CNYKQTSV
3,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,-0.164924,train,False,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,DNYKQTSV
4,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,-0.150309,train,False,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,ENYKQTSV


## Raw Count File Checks

The raw trans files should encode a 50-aa PDZ half followed by an 8-aa CRIPT peptide in `aa_seq`. Stop characters can occur and are handled after reconstruction.

In [4]:
def raw_file_summary(df: pd.DataFrame, source: str) -> dict[str, object]:
    aa_lengths = df["aa_seq"].str.len()
    return {
        "source": source,
        "rows": len(df),
        "unique_aa_seq": df["aa_seq"].nunique(),
        "min_aa_len": int(aa_lengths.min()),
        "median_aa_len": float(aa_lengths.median()),
        "max_aa_len": int(aa_lengths.max()),
        "rows_with_stop": int(df["aa_seq"].str.contains("*", regex=False).sum()),
        "duplicate_aa_seq_rows": int(df["aa_seq"].duplicated(keep=False).sum()),
        "total_counts": int(df[COUNT_COLUMNS].to_numpy().sum()),
    }

raw_summary = pd.DataFrame(
    [
        raw_file_summary(trans1_raw, "trans-1"),
        raw_file_summary(trans2_raw, "trans-2"),
    ]
)

display(raw_summary)
# We can also display the original raw data for trans-1 and trans-2 to visually check the structure
display(trans1_raw.head(3))
display(trans2_raw.head(3))

,source,rows,unique_aa_seq,min_aa_len,median_aa_len,max_aa_len,rows_with_stop,duplicate_aa_seq_rows,total_counts
0,trans-1,158655,150769,58,58.0,58,6709,9016,43698264
1,trans-2,159511,151624,58,58.0,58,6756,9017,34913614


,input1_e1_s0_bNA_count,output1A_e1_s1_b1_count,input2_e2_s0_bNA_count,output2A_e2_s1_b1_count,input3_e3_s0_bNA_count,output3A_e3_s1_b1_count,aa_seq
0,18,104,27,187,6,39,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...
1,6,35,17,80,12,38,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...
2,34,107,18,58,18,38,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...


,input1_e1_s0_bNA_count,output1A_e1_s1_b1_count,input2_e2_s0_bNA_count,output2A_e2_s1_b1_count,input3_e3_s0_bNA_count,output3A_e3_s1_b1_count,aa_seq
0,12,0,20,9,26,2,KRKGDQILSVNGVDLRNASHEQAAIALKNAGQTVTIIAQYKPEEYS...
1,22,4,23,1,23,3,KRKGDQILSVNGVDLRNASHEQAAIALKNAGQTVTIIAQYKPEEYS...
2,32,2,22,3,34,1,KRKGDQILSVNGVDLRNASHEQAAIALKNAGQTVTIIAQYKPEEYS...


## Reconstruct FLIP-Style Sequences

Apply the 50-aa wild-type completion rule to each trans file, concatenate both sources, and group by the normalized `PDZ:CRIPT` sequence used for FLIP merging.

In the code below, `raw_sequence` is the direct reconstruction before stop truncation. `sequence`, `pdz_sequence`, and `cript_sequence` are the stop-truncated values. Rows such as `:` are possible when both sides truncate to empty strings because a stop character appears at the first position; those rows are kept for accounting but excluded from the main grouped preview.

In [5]:
trans1_reconstructed = reconstruct_trans_sequences(trans1_raw, "trans-1")
trans2_reconstructed = reconstruct_trans_sequences(trans2_raw, "trans-2")
reconstructed_rows = pd.concat([trans1_reconstructed, trans2_reconstructed], ignore_index=True)

reconstructed_counts = (
    reconstructed_rows.groupby("sequence", as_index=False)
    .agg(
        {
            **{column: "sum" for column in COUNT_COLUMNS},
            "source": lambda values: ",".join(sorted(set(values))),
            "aa_seq": "size",
            "raw_sequence": pd.Series.nunique,
        }
    )
    .rename(columns={"aa_seq": "source_rows", "raw_sequence": "raw_sequence_count"})
)
reconstructed_counts[["pdz_sequence", "cript_sequence"]] = reconstructed_counts["sequence"].str.split(":", n=1, expand=True)

split_matches_sequence = reconstructed_counts["sequence"].eq(
    reconstructed_counts["pdz_sequence"] + ":" + reconstructed_counts["cript_sequence"]
)
if not split_matches_sequence.all():
    raise AssertionError("Split PDZ/CRIPT columns do not reconstruct the normalized sequence key.")

has_nonempty_pdz_and_cript = reconstructed_counts["pdz_sequence"].ne("") & reconstructed_counts["cript_sequence"].ne("")

print(
    "`raw_sequence` is the direct reconstruction before stop truncation; "
    "`sequence` is the normalized, stop-truncated FLIP merge key."
)
display(
    reconstructed_rows[
        [
            "source",
            "aa_seq",
            "raw_sequence",
            "sequence",
            "pdz_sequence",
            "cript_sequence",
        ]
    ].head(10)
)
display(
    reconstructed_counts.loc[
        has_nonempty_pdz_and_cript,
        [
            "sequence",
            "pdz_sequence",
            "cript_sequence",
            "source",
            "source_rows",
            "raw_sequence_count",
            *COUNT_COLUMNS,
        ],
    ].head(10)
)
display(
    reconstructed_counts.loc[
        ~has_nonempty_pdz_and_cript,
        ["sequence", "pdz_sequence", "cript_sequence", "source", "source_rows", "raw_sequence_count"],
    ].head(10)
)

`raw_sequence` is the direct reconstruction before stop truncation; `sequence` is the normalized, stop-truncated FLIP merge key.


,source,aa_seq,raw_sequence,sequence,pdz_sequence,cript_sequence
0,trans-1,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KKYKQTSV
1,trans-1,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KNKKQTSV
2,trans-1,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KNNKQTSV
3,trans-1,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KNTKQTSV
4,trans-1,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KNIKQTSV
5,trans-1,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KNMKQTSV
6,trans-1,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KNQKQTSV
7,trans-1,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KNHKQTSV
8,trans-1,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KNPKQTSV
9,trans-1,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KNRKQTSV


,sequence,pdz_sequence,cript_sequence,source,source_rows,raw_sequence_count,input1_e1_s0_bNA_count,input2_e2_s0_bNA_count,input3_e3_s0_bNA_count,output1A_e1_s1_b1_count,output2A_e2_s1_b1_count,output3A_e3_s1_b1_count
156,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,ANYKQTSV,trans-1,1,1,1,4,14,7,34,16
157,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,CNYKQTSV,trans-1,1,1,6,4,2,7,22,5
158,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,DNYKQTSV,trans-1,1,1,3,4,4,6,14,10
159,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,ENYKQTSV,trans-1,1,1,1,7,9,9,23,8
160,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,FNYKQTSV,trans-1,1,1,2,10,6,6,45,13
161,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,GNYKQTSV,trans-1,1,1,1,7,5,12,42,13
162,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,HNYKQTSV,trans-1,1,1,5,7,5,10,34,19
163,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,INYKQTSV,trans-1,1,1,15,5,8,19,52,15
164,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,K,trans-1,1,1,4,4,3,0,0,0
165,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,AEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,KAYKQTSV,trans-1,1,1,4,8,5,18,43,26


,sequence,pdz_sequence,cript_sequence,source,source_rows,raw_sequence_count
0,:,,,trans-1,1,1
1,:ANYKQTSV,,ANYKQTSV,trans-1,1,1
2,:CNYKQTSV,,CNYKQTSV,trans-1,1,1
3,:DNYKQTSV,,DNYKQTSV,trans-1,1,1
4,:ENYKQTSV,,ENYKQTSV,trans-1,1,1
5,:FNYKQTSV,,FNYKQTSV,trans-1,1,1
6,:GNYKQTSV,,GNYKQTSV,trans-1,1,1
7,:HNYKQTSV,,HNYKQTSV,trans-1,1,1
8,:INYKQTSV,,INYKQTSV,trans-1,1,1
9,:K,,K,trans-1,1,1


## Reconstruction Diagnostics

These tables show how many rows collapse during grouping, how often stop truncation changes a sequence, and where each reconstructed sequence came from.

In [6]:
reconstruction_summary = pd.DataFrame(
    [
        {
            "metric": "raw reconstructed rows before grouping",
            "value": len(reconstructed_rows),
        },
        {
            "metric": "unique reconstructed sequences after grouping",
            "value": reconstructed_counts["sequence"].nunique(),
        },
        {
            "metric": "rows changed by stop truncation",
            "value": int(
                (
                    reconstructed_rows["pdz_sequence_raw"].ne(reconstructed_rows["pdz_sequence"])
                    | reconstructed_rows["cript_sequence_raw"].ne(reconstructed_rows["cript_sequence"])
                ).sum()
            ),
        },
        {
            "metric": "sequences represented by both trans files",
            "value": int(reconstructed_counts["source"].eq("trans-1,trans-2").sum()),
        },
    ]
)

source_breakdown = reconstructed_counts["source"].value_counts().rename_axis("source").reset_index(name="unique_sequences")

display(reconstruction_summary)
display(source_breakdown)
display(reconstructed_counts.head(10))

,metric,value
0,raw reconstructed rows before grouping,318166
1,unique reconstructed sequences after grouping,302237
2,rows changed by stop truncation,13465
3,sequences represented by both trans files,156


,source,unique_sequences
0,trans-2,151468
1,trans-1,150613
2,"trans-1,trans-2",156


,sequence,input1_e1_s0_bNA_count,input2_e2_s0_bNA_count,input3_e3_s0_bNA_count,output1A_e1_s1_b1_count,output2A_e2_s1_b1_count,output3A_e3_s1_b1_count,source,source_rows,raw_sequence_count,pdz_sequence,cript_sequence
0,:,5,10,7,0,0,0,trans-1,1,1,,
1,:ANYKQTSV,12,22,13,0,0,1,trans-1,1,1,,ANYKQTSV
2,:CNYKQTSV,4,5,10,0,1,2,trans-1,1,1,,CNYKQTSV
3,:DNYKQTSV,12,14,9,0,0,0,trans-1,1,1,,DNYKQTSV
4,:ENYKQTSV,9,24,13,0,0,1,trans-1,1,1,,ENYKQTSV
5,:FNYKQTSV,10,6,4,0,1,0,trans-1,1,1,,FNYKQTSV
6,:GNYKQTSV,7,15,6,0,0,2,trans-1,1,1,,GNYKQTSV
7,:HNYKQTSV,13,8,3,0,1,0,trans-1,1,1,,HNYKQTSV
8,:INYKQTSV,12,11,14,0,2,0,trans-1,1,1,,INYKQTSV
9,:K,19,6,13,0,0,0,trans-1,1,1,,K


## FLIP Split Merge

This is the main reconciliation check: every sequence in the FLIP split file should be found in the reconstructed count table. Reconstructed-only rows are expected because the raw count universe is larger than the public split subset.

In [7]:
merged_flip = flip_split.merge(reconstructed_counts, on="sequence", how="left", indicator=True, suffixes=("_flip", "_counts"))
matched_flip = merged_flip["_merge"].eq("both")
reconstructed_on_flip = reconstructed_counts["sequence"].isin(flip_split["sequence"])

coverage_summary = pd.DataFrame(
    [
        {"metric": "trans-1 raw rows", "value": len(trans1_raw)},
        {"metric": "trans-2 raw rows", "value": len(trans2_raw)},
        {"metric": "reconstructed raw rows", "value": len(reconstructed_rows)},
        {"metric": "unique reconstructed sequences", "value": reconstructed_counts["sequence"].nunique()},
        {"metric": "FLIP split rows", "value": len(flip_split)},
        {"metric": "FLIP split rows matched to reconstructed counts", "value": int(matched_flip.sum())},
        {"metric": "FLIP split row coverage", "value": matched_flip.mean()},
        {"metric": "reconstructed sequences present in FLIP split", "value": int(reconstructed_on_flip.sum())},
        {"metric": "reconstructed sequence coverage on FLIP split", "value": reconstructed_on_flip.mean()},
    ]
)

merge_by_split = (
    merged_flip.assign(matched=matched_flip)
    .groupby(["set", "validation"], dropna=False)
    .agg(
        flip_rows=("sequence", "size"),
        matched_rows=("matched", "sum"),
        coverage=("matched", "mean"),
    )
    .reset_index()
)

count_columns_with_metadata = [
    "sequence",
    "set",
    "validation",
    "target",
    "source",
    "source_rows",
    *COUNT_COLUMNS,
]
matched_flip_counts = merged_flip.loc[matched_flip, count_columns_with_metadata].copy()
missing_flip_sequences = merged_flip.loc[~matched_flip, ["sequence", "set", "validation", "target"]].copy()
reconstructed_only_sequences = reconstructed_counts.loc[
    ~reconstructed_on_flip,
    ["sequence", "source", "source_rows", *COUNT_COLUMNS],
].copy()

display(coverage_summary)
display(merge_by_split)
display(matched_flip_counts.head(10))
display(missing_flip_sequences.head(20))
display(reconstructed_only_sequences.head(20))

,metric,value
0,trans-1 raw rows,158655.000000
1,trans-2 raw rows,159511.000000
2,reconstructed raw rows,318166.000000
3,unique reconstructed sequences,302237.000000
4,FLIP split rows,734.000000
5,FLIP split rows matched to reconstructed counts,734.000000
6,FLIP split row coverage,1.000000
7,reconstructed sequences present in FLIP split,734.000000
8,reconstructed sequence coverage on FLIP split,0.002429


,set,validation,flip_rows,matched_rows,coverage
0,test,False,579,579,1.0
1,train,False,124,124,1.0
2,train,True,31,31,1.0


,sequence,set,validation,target,source,source_rows,input1_e1_s0_bNA_count,input2_e2_s0_bNA_count,input3_e3_s0_bNA_count,output1A_e1_s1_b1_count,output2A_e2_s1_b1_count,output3A_e3_s1_b1_count
0,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,train,False,-0.834694,"trans-1,trans-2",52,9211,10730,12622,279,389,225
1,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,train,False,-0.085543,"trans-1,trans-2",52,13796,14923,18563,43804,64473,28716
2,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,train,False,-0.057949,"trans-1,trans-2",52,7813,8778,10565,26949,43673,18963
3,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,train,False,-0.164924,"trans-1,trans-2",52,12233,13383,16150,23721,38010,16881
4,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,train,False,-0.150309,"trans-1,trans-2",52,13566,14215,17838,25483,41889,19612
5,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,train,False,-0.101245,"trans-1,trans-2",52,10273,11196,13545,24267,42403,19417
6,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,train,False,-0.040958,"trans-1,trans-2",52,12248,12660,16481,41739,68805,32123
7,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,train,True,-0.053394,"trans-1,trans-2",52,12173,12741,15737,41260,62722,28789
8,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,train,True,-0.054483,"trans-1,trans-2",52,12692,13904,17034,40996,66353,30602
9,GEEDIPREPRRIVIHRGSTGLGFNIVGGEDGEGIFISFILAGGPAD...,train,False,-0.866563,"trans-1,trans-2",52,9895,11336,13174,452,722,393


,sequence,set,validation,target


,sequence,source,source_rows,input1_e1_s0_bNA_count,input2_e2_s0_bNA_count,input3_e3_s0_bNA_count,output1A_e1_s1_b1_count,output2A_e2_s1_b1_count,output3A_e3_s1_b1_count
0,:,trans-1,1,5,10,7,0,0,0
1,:ANYKQTSV,trans-1,1,12,22,13,0,0,1
2,:CNYKQTSV,trans-1,1,4,5,10,0,1,2
3,:DNYKQTSV,trans-1,1,12,14,9,0,0,0
4,:ENYKQTSV,trans-1,1,9,24,13,0,0,1
5,:FNYKQTSV,trans-1,1,10,6,4,0,1,0
6,:GNYKQTSV,trans-1,1,7,15,6,0,0,2
7,:HNYKQTSV,trans-1,1,13,8,3,0,1,0
8,:INYKQTSV,trans-1,1,12,11,14,0,2,0
9,:K,trans-1,1,19,6,13,0,0,0


## Optional CIS / CRIPT Sanity Check

The local Bruce Azure CIS file has `aa_seq` values that correspond to CRIPT peptide variants, not full `PDZ:CRIPT` strings. This optional check compares those peptide strings against the CRIPT side of the FLIP split and reconstructed trans table.

In [8]:
if cis_path.exists():
    cis_aa = pd.read_csv(cis_path, sep="\t", usecols=["aa_seq"])["aa_seq"].dropna().astype(str)
    cis_unique = pd.Index(cis_aa.map(truncate_stop).unique(), name="cript_sequence")
    flip_cript = pd.Index(flip_split["cript_sequence"].dropna().unique(), name="cript_sequence")
    reconstructed_cript = pd.Index(reconstructed_counts["cript_sequence"].dropna().unique(), name="cript_sequence")

    cis_summary = pd.DataFrame(
        [
            {"metric": "unique CIS aa_seq after stop truncation", "value": len(cis_unique)},
            {"metric": "unique FLIP CRIPT sequences", "value": len(flip_cript)},
            {"metric": "unique reconstructed CRIPT sequences", "value": len(reconstructed_cript)},
            {"metric": "CIS peptides present in FLIP CRIPT", "value": int(cis_unique.isin(flip_cript).sum())},
            {"metric": "FLIP CRIPT peptides present in CIS", "value": int(flip_cript.isin(cis_unique).sum())},
            {"metric": "CIS peptides present in reconstructed trans", "value": int(cis_unique.isin(reconstructed_cript).sum())},
        ]
    )

    display(cis_summary)
    display(pd.DataFrame({"flip_cript_missing_from_cis": flip_cript[~flip_cript.isin(cis_unique)][:20]}))
else:
    print(f"Skipping CIS peptide check because {cis_path.name} is not present.")

,metric,value
0,unique CIS aa_seq after stop truncation,10141
1,unique FLIP CRIPT sequences,155
2,unique reconstructed CRIPT sequences,156
3,CIS peptides present in FLIP CRIPT,155
4,FLIP CRIPT peptides present in CIS,155
5,CIS peptides present in reconstructed trans,156


,flip_cript_missing_from_cis


In [10]:
pd.read_csv(cis_path, sep="\t")

,nt_seq,aa_seq,WT,STOP,STOP_readthrough,Nham_nt,Nham_aa,Nmut_codons,input1_e1_s0_bNA_count,input2_e2_s0_bNA_count,input3_e3_s0_bNA_count,output1A_e1_s1_b1_count,output2A_e2_s1_b1_count,output3A_e3_s1_b1_count,barcode_valid,indel,constant_region,permitted,too_many_substitutions,mixed_substitutions
0,aaaaacaagaagaagacatctgtc,KNKKKTSV,NaN,False,False,4,2,2,388,456,265,341,294,27,True,False,True,True,False,False
1,aaaaacaagaagaatacatctgtc,KNKKNTSV,NaN,False,False,4,2,2,214,75,222,43,26,361,True,False,True,True,False,False
2,aaaaacaagaagacgacatctgtc,KNKKTTSV,NaN,False,False,5,2,2,706,1016,697,1374,1471,1616,True,False,True,True,False,False
3,aaaaacaagaagactacatctgtc,KNKKTTSV,NaN,False,False,5,2,2,10,20,58,2,7,161,True,False,True,True,False,False
4,aaaaacaagaagaggacatctgtc,KNKKRTSV,NaN,False,False,5,2,2,686,455,630,439,107,335,True,False,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26202,ttttcttacaagcaaacatctgtc,FSYKQTSV,NaN,False,False,6,2,2,463,205,364,1694,457,1150,True,False,True,True,False,False
26203,ttttggtacaagcaaacatctgtc,FWYKQTSV,NaN,False,False,6,2,2,15,3,19,2,1,9,True,False,True,True,False,False
26204,ttttgttacaagcaaacatctgtc,FCYKQTSV,NaN,False,False,6,2,2,0,0,0,1,0,0,True,False,True,True,False,False
26205,tttttgtacaagcaaacatctgtc,FLYKQTSV,NaN,False,False,6,2,2,241,322,64,567,361,42,True,False,True,True,False,False


## Optional Export

Set `write_outputs = True` to save the grouped reconstructed count table and the FLIP split table with attached counts. The default keeps the notebook read-only.

In [ ]:
write_outputs = False
output_dir = repo_root / "results" / "pdz_trans_reconciliation"

if write_outputs:
    output_dir.mkdir(parents=True, exist_ok=True)
    reconstructed_counts.to_csv(output_dir / "pdz_trans_reconstructed_counts.csv", index=False)
    matched_flip_counts.to_csv(output_dir / "pdz_trans_flip_split_with_counts.csv", index=False)
    coverage_summary.to_csv(output_dir / "pdz_trans_flip_coverage_summary.csv", index=False)
    print(f"Wrote outputs to {output_dir}")
else:
    print("write_outputs is False; no files written.")
    display(
        pd.DataFrame(
            {
                "available_output": [
                    "pdz_trans_reconstructed_counts.csv",
                    "pdz_trans_flip_split_with_counts.csv",
                    "pdz_trans_flip_coverage_summary.csv",
                ],
                "rows_if_written": [
                    len(reconstructed_counts),
                    len(matched_flip_counts),
                    len(coverage_summary),
                ],
            }
        )
    )